## **Race Data Analysis**

##### **Imports**

In [ ]:
import pandas as pd
import plotly.express as px
from scipy import stats

##### **Load Clean Laps**

- Filter lap data to only include laps without any track incidents, pit laps, or terminal laps
- Change **`LapTime`** and **`SectorTime`** to be seconds for later calculation and comparison
- Exclude columns not directly involved in Lap / Sector time analysis

In [ ]:
time_cols = ['LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time']
    
analysis_cols = [
    'Year', 'EventName', 'LapNumber', 'Driver', 'Team', 'Position', 
    'LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 
    'Rainfall', 'AirTemp', 'Humidity', 'Pressure', 'TrackTemp',
    'WindSpeed', 'WindDirection'
]

# Load race data
race_data = pd.read_pickle('../data/f1_lap_weather_data.pkl')

# Convert time columns to seconds
for col in time_cols:
    race_data[col] = race_data[col].dt.total_seconds()

clean_laps = (
    race_data.loc[
        (race_data['TrackStatus'] == 1)
        & (race_data['IsPitLap'] == False)
        & (race_data['IsTerminalLap'] == False),
        analysis_cols
    ]
)

clean_laps.head()
    

##### **Adverse Weather Races**

Using the weather context gained from the Open-Meteo analysis and the existing weather features in FastF1 we can identify which races drivers faced adverse weather conditions. 

The Open-Meteo analysis highlights several races with adverse weather:
- **German Grand Prix 2019** - Most rain (5-hr total)
- **Azerbaijan Grand Prix - 2018** - Highest wind gust
- **Spanish Grand Prix - 2022** - Hottest race start

We can perform a similair analysis using the **`Rainfall`** boolean from FastF1 to understand the percentage of laps that drivers faced rainy conditions. These environmental factors are often associated with difficult driving conditions, limiting driver visibility and decreasing track grip. 

In [ ]:
# Filter races that incurred rainfall
rainy_races = (
    clean_laps.copy()
    .groupby(['Year', 'EventName'], as_index=False)
    .agg(
        PercentRainfall=('Rainfall', lambda x: x.mean() * 100),
        TotalLaps=('Rainfall', 'count'),
        RainLaps=('Rainfall', 'sum')
    )
    .query('RainLaps > 0 and TotalLaps > 100')
    .sort_values('PercentRainfall', ascending=False)
    .reset_index(drop=True)
)

rainy_races.sort_values('PercentRainfall', ascending=False)

In [ ]:
# Reverse order for visual
rainy_races = rainy_races.sort_values('PercentRainfall', ascending=True)

# Add year + event name for clarity in visual
rainy_races['EventNameYear'] = rainy_races['Year'].astype(str) + ' ' + rainy_races['EventName'].str.replace('Grand Prix', 'GP')

# Create a bar chart for rainfall percentage
fig = px.bar(
    rainy_races,
    x='EventNameYear',
    y='PercentRainfall',
    orientation='v',
    hover_data=['RainLaps', 'TotalLaps'],
    color_discrete_sequence=["#C00000"]
)

fig.update_layout(
    xaxis_title={
        "text": "Event Name",
        "font": {
            "size": 15.5,
            "family": "Calibri",
            "color": "#000"
        }
    },
    yaxis_title={
        "text": "Percent of Laps with Rain",
        "font": {
            "size": 15.5,
            "family": "Calibri",
            "color": "#000"
        }
    },
    title={
        "text": "Top Rainiest Races (2018-2025)",
        "font": {
            "size": 18,
            "family": "Calibri Black",
            "color": "#000"
        }
    },
    plot_bgcolor="#FFF",
    margin=dict(t=100, l=100, r=100),
    width=1200
)

fig.update_xaxes(
    showline=True,
    linecolor="#000",
    linewidth=1.5
)

fig.update_yaxes(
    showline=True,
    linecolor="#000",
    linewidth=1.5,
    gridcolor="#d9d9d9"
)

fig.show()

##### **Lap / Sector Time Comparison**

- **`LapTimeDiff`** represents the different in a given driver's lap time and the median lap time of all drivers during that race
- **`Sector(1-3)TimeDiff`** represents the different in a given driver's sector time and the median sector time of all drivers during that lap

In [ ]:
# Calculate median lap time
median_laps = (
    clean_laps
    .groupby(['Year', 'EventName', 'LapNumber'], as_index=False)
    [time_cols]
    .agg('median')
    .rename({
        'LapTime': 'MedianLapTime',
        'Sector1Time': 'MedianSector1Time',
        'Sector2Time': 'MedianSector2Time',
        'Sector3Time': 'MedianSector3Time'
    }, axis=1)
)

# Merge clean laps with median
clean_laps = clean_laps.merge(median_laps, on=['Year', 'EventName', 'LapNumber'])

# Calculate time difference for each lap / sector
clean_laps['LapTimeDiff'] = clean_laps['LapTime'] - clean_laps['MedianLapTime']
clean_laps['Sector1TimeDiff'] = clean_laps['Sector1Time'] - clean_laps['MedianSector1Time']
clean_laps['Sector2TimeDiff'] = clean_laps['Sector2Time'] - clean_laps['MedianSector2Time']
clean_laps['Sector3TimeDiff'] = clean_laps['Sector3Time'] - clean_laps['MedianSector3Time']

clean_laps = clean_laps.drop(columns=['MedianLapTime', 'MedianSector1Time', 'MedianSector2Time', 'MedianSector3Time'])

clean_laps[['Year', 'EventName', 'Driver', 'LapNumber', 'LapTimeDiff', 'Sector1TimeDiff', 'Sector2TimeDiff', 'Sector3TimeDiff', 'Rainfall']].head()

##### **Performance Impact of Rainfall**

While it is generally understood that rainfall negatively impacts lap performance, we can still evaluate the strength and significance of this relationship using a t-test. We can also compare the mean performance differences observed during rainy and dry laps to better understand how adverse weather conditions affect drivers. By comparing these average differences, we can estimate the performance penalty associated with racing in rainy conditions.

- **Null Hypothesis:** There is no difference in driver performance between rainy and dry conditions.
- **Alternative Hypothesis:** Driver performance differs between rainy and dry conditions.

**Note:** One possible explanation for the non-significant Sector 1 results is the increased variability associated with race starts. Because Sector 1 experiences the highest levels of traffic and driver interaction, incidents such as spins, collisions, and defensive maneuvers may have a greater impact on sector times than weather conditions alone.

In [ ]:
# Filter times for races with rain
rainy_race_times = clean_laps.merge(
    rainy_races[['Year', 'EventName']].drop_duplicates(),
    on=['Year', 'EventName'],
    how='inner'
)

rainy_race_results = []

for col in ['LapTimeDiff', 'Sector1TimeDiff', 'Sector2TimeDiff', 'Sector3TimeDiff']:
    # Seperate into rain and dry diffs
    rain_diffs = rainy_race_times.loc[
        rainy_race_times['Rainfall'] == True, col
    ].dropna()

    dry_diffs = rainy_race_times.loc[
        rainy_race_times['Rainfall'] == False, col
    ].dropna()
    
    # Check for a raltionship between rain and lap times
    t_stat, p_value = stats.ttest_ind(rain_diffs, dry_diffs, equal_var=False)
    
    rainy_race_results.append({
        'Metric': col,
        'AvgRainDiff': rain_diffs.mean(),
        'AvgDryDiff': dry_diffs.mean(),
        'RainPenalty': rain_diffs.mean() - dry_diffs.mean(),
        'TStat': t_stat,
        'PValue': p_value,
        'RainLaps': len(rain_diffs),
        'DryLaps': len(dry_diffs)
    })
    
rainy_race_results = pd.DataFrame(rainy_race_results)
rainy_race_results

##### **Driver Performance in Adverse Weather Conditions**

Using the **`LapTimeDiff`** we can analyze the individual performance of drivers during adverse weather conditions. Drivers who are able to adapt quickly to changing conditions, manage tire grip effectively, and maintain consistency may gain a competitive advantage over the rest of the field.

- **`RainPerformanceGain`** - represents the average change in driver performance during rainy conditions relative to dry conditions. Positive values indicate the driver performed better relative to the field median in the rain, while negative values indicate worse performance.
- **`MIN_RAIN_LAPS`** - the minimum number of laps a driver must complete in the rain to be considered in the analysis

In [ ]:
MIN_RAIN_LAPS = 75

# Driver performance in rainy laps
rain_performance = (
    clean_laps.loc[clean_laps['Rainfall'] == True]
    .groupby('Driver')['LapTimeDiff']
    .agg(AvgRainDiff='mean', RainLapCount='count')
    .reset_index()
)

# Driver performance in dry laps
dry_performance = (
    clean_laps.loc[clean_laps['Rainfall'] == False]
    .groupby('Driver')['LapTimeDiff']
    .agg(AvgDryDiff='mean', DryLapCount='count')
    .reset_index()
)

weather_performance = rain_performance.merge(
    dry_performance,
    on='Driver'
)

# Calculate performance gain
weather_performance['RainPerformanceGain'] = (
    weather_performance['AvgDryDiff']
    - weather_performance['AvgRainDiff']
)

# Filter out drivers with less than 75 rainy laps
weather_performance = (
    weather_performance
    .loc[weather_performance['RainLapCount'] > MIN_RAIN_LAPS]
).round(3)

weather_performance.head()

In [ ]:
# Select top performers
top_10_rain_pace_drivers = weather_performance.sort_values('AvgRainDiff').head(10)
top_10_rain_pace_drivers = top_10_rain_pace_drivers.sort_values('AvgRainDiff', ascending=False)

# Create a bar chart comparing driver performance
fig = px.bar(
    top_10_rain_pace_drivers,
    x='Driver',
    y='AvgRainDiff',
    orientation='v',
    hover_data=['RainLapCount', 'DryLapCount'],
    color_discrete_sequence=["#C00000"]
)

fig.update_layout(
    xaxis_title={
        "text": "Driver",
        "font": {
            "size": 15.5,
            "family": "Calibri",
            "color": "#000"
        }
    },
    yaxis_title={
        "text": "Average Seconds",
        "font": {
            "size": 15.5,
            "family": "Calibri",
            "color": "#000"
        }
    },
    title={
        "text": "Top 10 Rainy Weather Drivers",
        "font": {
            "size": 18,
            "family": "Calibri Black",
            "color": "#000"
        },
    },
    plot_bgcolor="#FFF",
    margin=dict(t=100, l=100, r=50, b=60),
    width=600,
    height=400
)

fig.update_xaxes(
    showline=True,
    linecolor="#000",
    linewidth=1.5,
    side="top"
)

fig.update_yaxes(
    showline=True,
    linecolor="#000",
    linewidth=1.5,
    gridcolor="#d9d9d9"
)

fig.show()

In [ ]:
# Rain performance gain data
rain_performance_gain = weather_performance.sort_values('RainPerformanceGain')

# Create a bar chart comparing driver performance
fig = px.bar(
    rain_performance_gain,
    x='Driver',
    y='RainPerformanceGain',
    orientation='v',
    hover_data=['RainLapCount', 'DryLapCount'],
    color_discrete_sequence=["#C00000"]
)

fig.update_layout(
    xaxis_title={
        "text": "Driver",
        "font": {
            "size": 15.5,
            "family": "Calibri",
            "color": "#000"
        }
    },
    yaxis_title={
        "text": "Performance Gain (sec)",
        "font": {
            "size": 15.5,
            "family": "Calibri",
            "color": "#000"
        }
    },
    title={
        "text": "Driver Performance Gain in the Rain",
        "font": {
            "size": 18,
            "family": "Calibri Black",
            "color": "#000"
        }
    },
    plot_bgcolor="#FFF",
    margin=dict(t=100, l=100),
    width=1000
)

fig.update_xaxes(
    showline=True,
    linecolor="#000",
    linewidth=1.5
)

fig.update_yaxes(
    showline=True,
    linecolor="#000",
    linewidth=1.5,
    gridcolor="#d9d9d9"
)

fig.add_hline(
    y=0,
    line_color='black'
)

fig.show()

##### **Team Performance in Adverse Weather Conditions**

To evaluate team performance in adverse weather conditions, we aggregate **`LapTimeDiff`** values at the team level and compare performance during rainy and dry laps. Similar to the driver analysis, this approach measures how closely each team's drivers perform relative to the field median under different weather conditions.

**Possible Outcomes:**
- The impact of vehicle design on rainy-weather performance
- The effect of team strategies for handling changing track conditions on rainy-weather performance

In [ ]:
MIN_TEAM_RAIN_LAPS = 150 # increased to account for two drivers

# Team performance in rainy laps
team_rain_performance = (
    clean_laps.loc[clean_laps['Rainfall'] == True]
    .groupby('Team')
    .agg(
        AvgRainDiff=('LapTimeDiff', 'mean'), 
        RainLapCount=('LapTimeDiff', 'count'), 
        DriverCountWet=('Driver', 'nunique')
    )
    .reset_index()
)

# Team performance in dry laps
team_dry_performance = (
    clean_laps.loc[clean_laps['Rainfall'] == False]
    .groupby('Team')
    .agg(
        AvgDryDiff=('LapTimeDiff', 'mean'), 
        DryLapCount=('LapTimeDiff', 'count'), 
        DriverCountDry=('Driver', 'nunique')
    )
    .reset_index()
)

team_weather_performance = team_rain_performance.merge(
    team_dry_performance,
    on='Team'
)

# Calculate performance gain
team_weather_performance['RainPerformanceGain'] = (
    team_weather_performance['AvgDryDiff']
    - team_weather_performance['AvgRainDiff']
)

# Filter out drivers with less than 75 rainy laps
team_weather_performance = (
    team_weather_performance
    .loc[team_weather_performance['RainLapCount'] > MIN_TEAM_RAIN_LAPS]
).round(3)

team_weather_performance

In [ ]:
# Rain performance gain data
team_rain_performance_gain = team_weather_performance.sort_values('RainPerformanceGain')

# Create a bar chart comparing driver performance
fig = px.bar(
    team_rain_performance_gain,
    x='Team',
    y='RainPerformanceGain',
    orientation='v',
    hover_data=['RainLapCount', 'DryLapCount'],
    color_discrete_sequence=["#C00000"]
)

fig.update_layout(
    xaxis_title={
        "text": "Team",
        "font": {
            "size": 15.5,
            "family": "Calibri",
            "color": "#000"
        }
    },
    yaxis_title={
        "text": "Performance Gain",
        "font": {
            "size": 15.5,
            "family": "Calibri",
            "color": "#000"
        }
    },
    title={
        "text": "Team Performance Gain in the Rain",
        "font": {
            "size": 18,
            "family": "Calibri Black",
            "color": "#000"
        }
    },
    plot_bgcolor="#FFF",
    margin=dict(t=100, l=100),
    width=700
)

fig.update_xaxes(
    showline=True,
    linecolor="#000",
    linewidth=1.5
)

fig.update_yaxes(
    showline=True,
    linecolor="#000",
    linewidth=1.5,
    gridcolor="#d9d9d9"
)

fig.add_hline(
    y=0,
    line_color='black'
)

fig.show()